In [ ]:
import sys
from pathlib import Path
import cv2
import pandas as pd
import matplotlib.pyplot as plt

# Ensure the src folder is accessible
PROJECT_ROOT = Path.cwd().resolve().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# Import centralized paths
from src.config import DATASET_ROOT

In [ ]:
CLASS_MAPPING = {"class_0_genuine": "genuine", "class_1_forged": "forged"}
VALID_EXTENSIONS = {".png", ".jpg", ".jpeg"}

dataset_records = []
unreadable_files = []

print(f"Scanning Dataset at: {DATASET_ROOT}")

for folder_name, class_name in CLASS_MAPPING.items():
    folder_path = DATASET_ROOT / folder_name

    if not folder_path.exists():
        print(f"WARNING: Missing folder {folder_path}")
        continue

    for file_path in folder_path.rglob("*"):
        if file_path.is_file() and file_path.suffix.lower() in VALID_EXTENSIONS:
            # Check if image is readable
            img = cv2.imread(str(file_path))
            if img is None:
                unreadable_files.append(file_path)
                continue

            h, w = img.shape[:2]

            dataset_records.append(
                {
                    "file_path": str(file_path),
                    "file_name": file_path.name,
                    "class_name": class_name,
                    "width": w,
                    "height": h,
                    "extension": file_path.suffix.lower(),
                }
            )

df_dataset = pd.DataFrame(dataset_records)

print("=" * 40)
print(f"Total readable images: {len(df_dataset)}")
print(f"Unreadable images: {len(unreadable_files)}")
print("=" * 40)

In [ ]:
class_counts = df_dataset["class_name"].value_counts()

print("Class Distribution:")
print(class_counts)

plt.figure(figsize=(6, 4))
class_counts.plot(kind="bar", color=["#4C72B0", "#C44E52"])
plt.title("Dataset Class Distribution")
plt.xlabel("Class")
plt.ylabel("Number of Images")
plt.xticks(rotation=0)
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
print("Image File Extensions:")
print(df_dataset["extension"].value_counts())
print("\n" + "=" * 40 + "\n")

resolution_counts = (
    df_dataset.groupby(["width", "height"])
    .size()
    .reset_index(name="count")
    .sort_values("count", ascending=False)
)

print("Top Image Resolutions:")
print(resolution_counts.head())

In [ ]:
def show_samples(df, class_name, n=4, display_width=500):
    sample_df = df[df["class_name"] == class_name].sample(
        n=min(n, len(df)), random_state=42
    )

    plt.figure(figsize=(16, 6))
    for i, (_, row) in enumerate(sample_df.iterrows(), start=1):
        img = cv2.imread(row["file_path"])
        h, w = img.shape[:2] # type: ignore
        scale = display_width / w
        img_small = cv2.resize(img, (display_width, int(h * scale))) # type: ignore

        plt.subplot(1, len(sample_df), i)
        plt.imshow(cv2.cvtColor(img_small, cv2.COLOR_BGR2RGB))
        plt.title(row["file_name"], fontsize=10)
        plt.axis("off")

    plt.tight_layout()
    plt.show()


# Show Genuine Samples
print("Genuine Stamps:")
show_samples(df_dataset, "genuine")

# Show Forged Samples
print("Forged Stamps:")
show_samples(df_dataset, "forged")